# WP3 Africa Hazard Ranking — Dataset Notebook 02  
## INFORM Risk (Mid 2025 v0.71) — country-level indicator extraction

**Goal (dataset-specific):**  
Extract a *country-level* set of **INFORM Risk** indicators for the WP3 Africa hazard ranking workflow, harmonised to the shared **long-format master contract**, and upsert into:

`data/intermediate/wp3_country_indicators_master.csv`

**Project constraints implemented here**
- Scope = your `ISO3_Regions.csv` list only (North Africa excluded by construction of that list; Réunion included).
- Hazards fixed at 9 + a special “All Hazards”.
- **Presence is NOT a scoring dimension**: it is handled as a **coverage output** only.
- This dataset is a **snapshot** (Mid 2025 release). Values are on a **0–10 scale** in the INFORM framework.

**Why INFORM matters for WP3**
INFORM Risk decomposes risk into three dimensions: **Hazard & Exposure**, **Vulnerability**, and **Lack of Coping Capacity**.  
In this WP3 framework we use a **subset** of INFORM outputs as agreed:
- Hazard-specific hazard/exposure indicators as **Scale proxies**
- Overall Hazard & Exposure (all hazards) as **Scale**
- Lack of Coping Capacity (all hazards) as **Cascade impacts proxy**

---
## Inputs (local)
- `data/raw/scope/ISO3_Regions.csv` (or fallback `/mnt/data/ISO3_Regions.csv`)
- `data/raw/INFORM/INFORM_Risk_Mid_2025_v071.xlsx` (or fallback `/mnt/data/INFORM_Risk_Mid_2025_v071.xlsx`)

## Outputs (local)
- `data/intermediate/inform_risk_extracted_2025_long.csv`
- `data/intermediate/inform_risk_coverage_2025.csv`
- upserted master: `data/intermediate/wp3_country_indicators_master.csv`

---
## Indicator mapping implemented (as provided by you)

**Hazard-specific → Scale**
- Drought / Scale → `Drought`
- Riverine Flooding Continued & Cluster / Scale → `River Flood`
- Storm Surge / Scale → `Coastal flood` (INFORM naming)
- Wind / Scale → `Tropical Cyclone` (proxy for wind-storm exposure)

**All Hazards**
- Scale → `HAZARD & EXPOSURE`
- Cascade impacts → `LACK OF COPING CAPACITY`

**Presence (coverage only, not appended)**
Presence flags are created as: value-not-missing for each hazard-specific column above.

---
## Notes / caveats 
- Using **Lack of Coping Capacity** as a proxy for “Cascade impacts” is an *operational choice* for WP3; it measures institutional/infrastructure capacity to cope and recover, not cascades directly.
- “Tropical Cyclone” is used as the closest INFORM proxy to **Wind** hazard category; document this in interpretation.

In [11]:
from pathlib import Path
import pandas as pd
import numpy as np

# -----------------------------
# Config — edit paths if needed
# -----------------------------
PROJECT_ROOT = Path(".").resolve()

# Inputs (repo-style)
COUNTRY_SCOPE_PATH = PROJECT_ROOT.parent / "data" / "raw" / "scope" / "ISO3_Regions.csv"
INFORM_XLSX_PATH   = PROJECT_ROOT.parent / "data" / "raw" / "INFORM" / "INFORM_Risk_Mid_2025_v071.xlsx"

# Fallbacks for quick local testing in sandboxes
FALLBACK_SCOPE = Path("/mnt/data/ISO3_Regions.csv")
FALLBACK_INFORM = Path("/mnt/data/INFORM_Risk_Mid_2025_v071.xlsx")

if not COUNTRY_SCOPE_PATH.exists() and FALLBACK_SCOPE.exists():
    COUNTRY_SCOPE_PATH = FALLBACK_SCOPE
if not INFORM_XLSX_PATH.exists() and FALLBACK_INFORM.exists():
    INFORM_XLSX_PATH = FALLBACK_INFORM

# Master path (your single source of truth)
MASTER_PATH = PROJECT_ROOT.parent / "data" / "intermediate" / "wp3_country_indicators_long.csv"
MASTER_PATH.parent.mkdir(parents=True, exist_ok=True)

# Outputs
OUT_DIR = PROJECT_ROOT.parent / "data" / "intermediate"
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2025  # INFORM "Mid 2025" edition used as snapshot year label (not an event window)

OUT_LONG     = OUT_DIR / "inform_risk_extracted_2025_long.csv"
OUT_COVERAGE = OUT_DIR / "inform_risk_coverage_2025.csv"

print("PROJECT_ROOT:", PROJECT_ROOT.parent)
print("COUNTRY_SCOPE_PATH:", COUNTRY_SCOPE_PATH, "exists:", COUNTRY_SCOPE_PATH.exists())
print("INFORM_XLSX_PATH:", INFORM_XLSX_PATH, "exists:", INFORM_XLSX_PATH.exists())
print("MASTER_PATH:", MASTER_PATH)

PROJECT_ROOT: C:\pipelines\sewa-multihazar
COUNTRY_SCOPE_PATH: C:\pipelines\sewa-multihazar\data\raw\scope\ISO3_Regions.csv exists: True
INFORM_XLSX_PATH: C:\pipelines\sewa-multihazar\data\raw\INFORM\INFORM_Risk_Mid_2025_v071.xlsx exists: True
MASTER_PATH: C:\pipelines\sewa-multihazar\data\intermediate\wp3_country_indicators_long.csv


## Global setup — shared long-format contract and upsert

This notebook writes indicators in the same contract used across all dataset notebooks and performs an **upsert** into the master CSV.

**Upsert key** (must be unique):  
`iso3 + hazard + dimension + source + indicator_id + year`

In [12]:
# Canonical enums
HAZARDS_9 = [
    "Drought",
    "Flash Flooding",
    "Riverine Flooding Continued & Cluster",
    "Heatwave",
    "Storm Surge",
    "Wind",
    "Thunderstorm",
    "Wildfires",
    "Dust",
]
DIMENSIONS_5 = ["Prevalence", "Scale", "Impact", "Cascade impacts", "Future relevance"]

SOURCE = "INFORM Risk"

LONG_COLUMNS = [
    "iso3","country_name","region","hazard","dimension","source",
    "indicator_id","indicator_name","value_raw","unit_raw",
    "year","time_window","notes"
]
UPSERT_KEY = ["iso3","hazard","dimension","source","indicator_id","year"]

def assert_contract(df: pd.DataFrame) -> None:
    missing_cols = [c for c in LONG_COLUMNS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Contract violation: missing columns {missing_cols}")

    bad_dims = sorted(set(df["dimension"]) - set(DIMENSIONS_5))
    if bad_dims:
        raise ValueError(f"Unknown dimensions {bad_dims}")

    allowed_haz = set(HAZARDS_9) | {"All Hazards"}
    bad_haz = sorted(set(df["hazard"]) - allowed_haz)
    if bad_haz:
        raise ValueError(f"Unknown hazards {bad_haz}")

    dup = df.duplicated(UPSERT_KEY, keep=False)
    if dup.any():
        raise ValueError("Duplicate UPSERT keys found. Check indicator_id or year assignment.")

def upsert_to_master(new_df: pd.DataFrame, master_path: Path) -> pd.DataFrame:
    assert_contract(new_df)

    if master_path.exists():
        master = pd.read_csv(master_path)
        for col in LONG_COLUMNS:
            if col not in master.columns:
                master[col] = np.nan
        master = master[LONG_COLUMNS]

        master_key = master[UPSERT_KEY].astype(str).agg("||".join, axis=1)
        new_key    = new_df[UPSERT_KEY].astype(str).agg("||".join, axis=1)
        master = master.loc[~master_key.isin(set(new_key))].copy()

        out = pd.concat([master, new_df[LONG_COLUMNS]], ignore_index=True)
    else:
        out = new_df[LONG_COLUMNS].copy()

    out.to_csv(master_path, index=False)
    return out

## Step 1 — Load scope list (ISO3 + region)

In [13]:
scope_raw = pd.read_csv(COUNTRY_SCOPE_PATH)
scope_raw.columns = [c.replace("\xa0"," ").strip() for c in scope_raw.columns]

scope = scope_raw.copy()
scope["Region"] = scope["Region"].replace({np.nan: None}).ffill()
scope["Region"] = scope["Region"].astype(str).str.replace("\xa0","", regex=False).str.strip()
scope["Country"] = scope["Country"].astype(str).str.replace("\xa0","", regex=False).str.strip()
scope["ISO3.Code"] = scope["ISO3.Code"].astype(str).str.strip().str.upper()

scope = scope.rename(columns={"Region":"region","Country":"country_name","ISO3.Code":"iso3"})
scope = scope[["iso3","country_name","region"]].drop_duplicates()

if scope["iso3"].duplicated().any():
    raise ValueError("Duplicate ISO3 in scope list.")

print("Scope countries:", len(scope))
scope.head()

Scope countries: 47


,iso3,country_name,region
0,NGA,Nigeria,West Africa
1,GHA,Ghana,West Africa
2,SEN,Senegal,West Africa
3,BFA,Burkina Faso,West Africa
4,MLI,Mali,West Africa


## Step 2 — Read INFORM Risk (Mid 2025) sheet and clean headers

In [14]:
SHEET = "INFORM Risk Mid 2025 (a-z)"

raw = pd.read_excel(INFORM_XLSX_PATH, sheet_name=SHEET, header=None)

headers = raw.iloc[1].tolist()   # names row
units   = raw.iloc[2].tolist()   # units row (not used programmatically, but useful for QA)
df = raw.iloc[3:].copy()
df.columns = headers

# Canonical columns
df = df.rename(columns={"COUNTRY":"country_inform","ISO3":"iso3"})
df["iso3"] = df["iso3"].astype(str).str.strip().str.upper()
df["country_inform"] = df["country_inform"].astype(str).str.strip()

# Keep only columns we plan to use (+ optional diagnostic)
keep_cols = [
    "iso3","country_inform",
    "HAZARD & EXPOSURE",
    "LACK OF COPING CAPACITY",
    "Drought",
    "River Flood",
    "Coastal flood",
    "Tropical Cyclone",
]
missing = [c for c in keep_cols if c not in df.columns]
if missing:
    raise ValueError(f"Expected columns not found in INFORM sheet: {missing}")

inform = df[keep_cols].copy()

# Convert to numeric (coerce)
for c in keep_cols[2:]:
    inform[c] = pd.to_numeric(inform[c], errors="coerce")

inform.head()

c:\Users\duruenaramirez\AppData\Local\miniforge3\envs\ibf-env\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():
c:\Users\duruenaramirez\AppData\Local\miniforge3\envs\ibf-env\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():


,iso3,country_inform,HAZARD & EXPOSURE,LACK OF COPING CAPACITY,Drought,River Flood,Coastal flood,Tropical Cyclone
3,AFG,Afghanistan,7.5,7.4,8.7,7.3,0.0,0.0
4,ALB,Albania,3.5,3.4,3.9,4.6,7.4,0.0
5,DZA,Algeria,3.0,4.3,2.6,3.6,0.7,0.0
6,AGO,Angola,4.2,6.9,3.7,3.9,3.5,0.0
7,ATG,Antigua and Barbuda,2.0,3.5,0.0,0.0,4.3,8.2


## Step 3 — Join to scope and report any missing ISO3

In [15]:
joined = scope.merge(inform, on="iso3", how="left")

missing_iso3 = joined.loc[joined["country_inform"].isna(), ["iso3","country_name","region"]]
print("Missing in INFORM for scope ISO3:", missing_iso3["iso3"].tolist())
missing_iso3

Missing in INFORM for scope ISO3: ['REU']


,iso3,country_name,region
46,REU,Réunion (France),Southern Africaincl. Indian Ocean Islands Country


## Step 4 — Build long-format indicator table (appendable to master)

In [16]:
INDICATORS = [
    # Hazard-specific scale proxies
    dict(hazard="Drought", dimension="Scale", col="Drought",
         indicator_id="INFORM.HAZEX.DROUGHT", indicator_name="Hazard & exposure: Drought"),
    dict(hazard="Riverine Flooding Continued & Cluster", dimension="Scale", col="River Flood",
         indicator_id="INFORM.HAZEX.RIVER_FLOOD", indicator_name="Hazard & exposure: River Flood"),
    dict(hazard="Storm Surge", dimension="Scale", col="Coastal flood",
         indicator_id="INFORM.HAZEX.COASTAL_FLOOD", indicator_name="Hazard & exposure: Coastal Flood (proxy for Storm Surge)"),
    dict(hazard="Wind", dimension="Scale", col="Tropical Cyclone",
         indicator_id="INFORM.HAZEX.TROPICAL_CYCLONE", indicator_name="Hazard & exposure: Tropical Cyclone (proxy for Wind)"),

    # All hazards
    dict(hazard="All Hazards", dimension="Scale", col="HAZARD & EXPOSURE",
         indicator_id="INFORM.DIM.HAZARD_EXPOSURE", indicator_name="INFORM dimension: Hazard & Exposure"),
    dict(hazard="All Hazards", dimension="Cascade impacts", col="LACK OF COPING CAPACITY",
         indicator_id="INFORM.DIM.LACK_COPING", indicator_name="INFORM dimension: Lack of Coping Capacity (proxy for cascade impacts)"),
]

rows = []
for spec in INDICATORS:
    tmp = joined[["iso3","country_name","region", spec["col"]]].copy()
    tmp = tmp.rename(columns={spec["col"]:"value_raw"})
    tmp["hazard"] = spec["hazard"]
    tmp["dimension"] = spec["dimension"]
    tmp["source"] = SOURCE
    tmp["indicator_id"] = spec["indicator_id"]
    tmp["indicator_name"] = spec["indicator_name"]
    tmp["unit_raw"] = "index_0_10"
    tmp["year"] = YEAR
    tmp["time_window"] = "annual_snapshot"
    tmp["notes"] = f"Extracted from INFORM Risk Mid 2025. Column='{spec['col']}'."
    rows.append(tmp)

long_df = pd.concat(rows, ignore_index=True)

# Drop missing values (coverage handled separately)
long_df = long_df.loc[~long_df["value_raw"].isna()].copy()

# Order columns
long_df = long_df[LONG_COLUMNS]

assert_contract(long_df)

print("Rows extracted:", len(long_df))
long_df.head(12)

Rows extracted: 276


,iso3,country_name,region,hazard,dimension,source,indicator_id,indicator_name,value_raw,unit_raw,year,time_window,notes
0,NGA,Nigeria,West Africa,Drought,Scale,INFORM Risk,INFORM.HAZEX.DROUGHT,Hazard & exposure: Drought,3.1,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='D...
1,GHA,Ghana,West Africa,Drought,Scale,INFORM Risk,INFORM.HAZEX.DROUGHT,Hazard & exposure: Drought,1.3,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='D...
2,SEN,Senegal,West Africa,Drought,Scale,INFORM Risk,INFORM.HAZEX.DROUGHT,Hazard & exposure: Drought,6.6,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='D...
3,BFA,Burkina Faso,West Africa,Drought,Scale,INFORM Risk,INFORM.HAZEX.DROUGHT,Hazard & exposure: Drought,6.3,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='D...
4,MLI,Mali,West Africa,Drought,Scale,INFORM Risk,INFORM.HAZEX.DROUGHT,Hazard & exposure: Drought,7.9,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='D...
5,NER,Niger,West Africa,Drought,Scale,INFORM Risk,INFORM.HAZEX.DROUGHT,Hazard & exposure: Drought,7.1,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='D...
6,CIV,Côte d'Ivoire,West Africa,Drought,Scale,INFORM Risk,INFORM.HAZEX.DROUGHT,Hazard & exposure: Drought,0.9,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='D...
7,BEN,Benin,West Africa,Drought,Scale,INFORM Risk,INFORM.HAZEX.DROUGHT,Hazard & exposure: Drought,0.9,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='D...
8,TGO,Togo,West Africa,Drought,Scale,INFORM Risk,INFORM.HAZEX.DROUGHT,Hazard & exposure: Drought,1.3,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='D...
9,SLE,Sierra Leone,West Africa,Drought,Scale,INFORM Risk,INFORM.HAZEX.DROUGHT,Hazard & exposure: Drought,0.9,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='D...


## Step 5 — Coverage

In [17]:
coverage_specs = [
    ("Drought", "Drought"),
    ("Riverine Flooding Continued & Cluster", "River Flood"),
    ("Storm Surge", "Coastal flood"),
    ("Wind", "Tropical Cyclone"),
]

cov_rows = []
for haz, col in coverage_specs:
    tmp = joined[["iso3","country_name","region", col]].copy()
    tmp["hazard"] = haz
    tmp["source"] = SOURCE
    tmp["present"] = ~tmp[col].isna()
    tmp["notes"] = f"Presence derived from INFORM column '{col}' not-missing."
    cov_rows.append(tmp[["iso3","country_name","region","hazard","source","present","notes"]])

coverage = pd.concat(cov_rows, ignore_index=True)
coverage.to_csv(OUT_COVERAGE, index=False)

coverage.groupby(["hazard","present"]).size().unstack(fill_value=0)

present,False,True
hazard,,
Drought,1,46
Riverine Flooding Continued & Cluster,1,46
Storm Surge,1,46
Wind,1,46


## Step 6 — Write dataset extract + upsert to master

In [18]:
long_df.to_csv(OUT_LONG, index=False)

master = upsert_to_master(long_df, MASTER_PATH)

print("Wrote:", OUT_LONG)
print("Wrote:", OUT_COVERAGE)
print("Updated master:", MASTER_PATH, "rows:", len(master))

master.tail(10)

Wrote: C:\pipelines\sewa-multihazar\data\intermediate\inform_risk_extracted_2025_long.csv
Wrote: C:\pipelines\sewa-multihazar\data\intermediate\inform_risk_coverage_2025.csv
Updated master: C:\pipelines\sewa-multihazar\data\intermediate\wp3_country_indicators_long.csv rows: 552


,iso3,country_name,region,hazard,dimension,source,indicator_id,indicator_name,value_raw,unit_raw,year,time_window,notes
542,NAM,Namibia,Southern Africaincl. Indian Ocean Islands Country,All Hazards,Cascade impacts,INFORM Risk,INFORM.DIM.LACK_COPING,INFORM dimension: Lack of Coping Capacity (pro...,4.7,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='L...
543,MOZ,Mozambique,Southern Africaincl. Indian Ocean Islands Country,All Hazards,Cascade impacts,INFORM Risk,INFORM.DIM.LACK_COPING,INFORM dimension: Lack of Coping Capacity (pro...,6.8,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='L...
544,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,All Hazards,Cascade impacts,INFORM Risk,INFORM.DIM.LACK_COPING,INFORM dimension: Lack of Coping Capacity (pro...,6.9,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='L...
545,MWI,Malawi,Southern Africaincl. Indian Ocean Islands Country,All Hazards,Cascade impacts,INFORM Risk,INFORM.DIM.LACK_COPING,INFORM dimension: Lack of Coping Capacity (pro...,5.9,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='L...
546,LSO,Lesotho,Southern Africaincl. Indian Ocean Islands Country,All Hazards,Cascade impacts,INFORM Risk,INFORM.DIM.LACK_COPING,INFORM dimension: Lack of Coping Capacity (pro...,6.3,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='L...
547,SWZ,Eswatini (Swaziland),Southern Africaincl. Indian Ocean Islands Country,All Hazards,Cascade impacts,INFORM Risk,INFORM.DIM.LACK_COPING,INFORM dimension: Lack of Coping Capacity (pro...,4.9,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='L...
548,MDG,Madagascar,Southern Africaincl. Indian Ocean Islands Country,All Hazards,Cascade impacts,INFORM Risk,INFORM.DIM.LACK_COPING,INFORM dimension: Lack of Coping Capacity (pro...,7.5,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='L...
549,MUS,Mauritius,Southern Africaincl. Indian Ocean Islands Country,All Hazards,Cascade impacts,INFORM Risk,INFORM.DIM.LACK_COPING,INFORM dimension: Lack of Coping Capacity (pro...,2.5,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='L...
550,SYC,Seychelles,Southern Africaincl. Indian Ocean Islands Country,All Hazards,Cascade impacts,INFORM Risk,INFORM.DIM.LACK_COPING,INFORM dimension: Lack of Coping Capacity (pro...,2.3,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='L...
551,COM,Comoros,Southern Africaincl. Indian Ocean Islands Country,All Hazards,Cascade impacts,INFORM Risk,INFORM.DIM.LACK_COPING,INFORM dimension: Lack of Coping Capacity (pro...,7.1,index_0_10,2025,annual_snapshot,Extracted from INFORM Risk Mid 2025. Column='L...


## Optional: “Extra” INFORM columns we are *not* using yet

Flag what else could be useful for other dimensions.
This cell displays the main dimension and category-level columns available in the sheet.
Nothing here is appended to the master unless you explicitly add it to `INDICATORS`.

In [19]:
# High-level candidates you might want later:
candidates = [
    "INFORM RISK",
    "Natural",
    "Human",
    "VULNERABILITY",
    "Socio-Economic", "Vulnerable Groups",
]
available = [c for c in candidates if c in df.columns]
df[["country_inform","iso3"] + available].head(10)

,country_inform,iso3,INFORM RISK,Natural,Human,VULNERABILITY,Vulnerable Groups
3,Afghanistan,AFG,7.7,5.7,8.7,8.2,8.3
4,Albania,ALB,3,5.5,0.7,2.2,2.5
5,Algeria,DZA,3.4,3.3,2.6,3,3.8
6,Angola,AGO,5.3,3,5.3,5.2,4.7
7,Antigua and Barbuda,ATG,2.1,3.7,0,1.4,1.3
8,Argentina,ARG,3.1,4.3,0.5,2.7,3.9
9,Armenia,ARM,3,3.7,1.3,3.7,5.4
10,Australia,AUS,2.4,4.7,0.5,2.1,3.6
11,Austria,AUT,2.4,2.5,0.2,3.2,5.3
12,Azerbaijan,AZE,4.1,4.1,1.6,4.8,6.6
